In [ ]:
# --- Cell 1: setup, paths and helper functions (self-contained) ---
from google.colab import drive
drive.mount('/content/drive')

import os, csv, gzip, re, urllib.request
import pandas as pd

PROJECT = "/content/drive/MyDrive/dlbcl-prognosis"
RAW = os.path.join(PROJECT, "data", "raw")
os.makedirs(RAW, exist_ok=True)

def geo_matrix_urls(gse):
    base = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{gse[:-3]}nnn/{gse}/matrix/"
    html = urllib.request.urlopen(base, timeout=60).read().decode()
    files = sorted(set(re.findall(r'href="([^"]*_series_matrix\.txt\.gz)"', html)))
    return [base + f for f in files]

def remote_size_mb(url):
    req = urllib.request.Request(url, method="HEAD")
    with urllib.request.urlopen(req, timeout=60) as r:
        return int(r.headers.get("Content-Length", 0)) / 1e6

def stream_header(url):
    lines = []
    with urllib.request.urlopen(url, timeout=300) as r, gzip.GzipFile(fileobj=r) as gz:
        for raw in gz:
            line = raw.decode("utf-8", errors="replace").rstrip("\r\n")
            if line.startswith("!series_matrix_table_begin"):
                break
            lines.append(line)
    return lines

def split_kv(v):
    if ": " in v:
        return v.rsplit(": ", 1)
    if ":" in v:
        return v.rsplit(":", 1)
    return "unlabeled", v

def parse_header(lines):
    rows = [next(csv.reader([l], delimiter="\t")) for l in lines if l.startswith("!Sample_")]
    n = len(rows[0]) - 1
    recs = [{} for _ in range(n)]
    keep = {"!Sample_geo_accession": "gsm", "!Sample_title": "title",
            "!Sample_platform_id": "platform", "!Sample_source_name_ch1": "source"}
    for row in rows:
        key, vals = row[0], row[1:]
        if key in keep:
            for i, v in enumerate(vals):
                recs[i][keep[key]] = v
        elif key == "!Sample_characteristics_ch1":
            for i, v in enumerate(vals):
                v = v.strip()
                if not v:
                    continue
                k, val = split_kv(v)
                k = k.strip()
                while k in recs[i]:
                    k += "_dup"
                recs[i][k] = val.strip()
    return pd.DataFrame(recs)

def series_title(lines):
    for l in lines:
        if l.startswith("!Series_title"):
            return l.split("\t", 1)[1].strip('"')
    return "?"

def summarize(df, max_vals=4, width=45):
    for col in df.columns:
        if col in ("gsm", "title"):
            continue
        s = df[col]
        top = "; ".join(f"{str(k)[:width]} ({v})" for k, v in s.value_counts().head(max_vals).items())
        print(f"  - {col[:60]} | missing={s.isna().sum()} | unique={s.nunique()} | {top}")

print("Helpers ready. Raw dir exists:", os.path.isdir(RAW))

In [ ]:
# --- Cell 2: check UNVERIFIED candidate cohorts for survival data in GEO metadata ---
# These IDs come from memory and are NOT yet verified; a failure here is itself informative.
CANDIDATES = ["GSE181063", "GSE87371", "GSE32918"]

for gse in CANDIDATES:
    print(f"\n########## {gse} ##########")
    try:
        urls = geo_matrix_urls(gse)
        if not urls:
            print("  no series-matrix files found")
            continue
        dfs = []
        for url in urls:
            lines = stream_header(url)
            fname = url.rsplit("/", 1)[-1]
            with open(os.path.join(RAW, fname.replace(".txt.gz", "_header.txt")), "w") as f:
                f.write("\n".join(lines))
            print(f"  {fname} | full size ~ {remote_size_mb(url):.0f} MB")
            print(f"  title: {series_title(lines)[:150]}")
            dfs.append(parse_header(lines))
        df = pd.concat(dfs, ignore_index=True)
        df.to_csv(os.path.join(RAW, f"{gse}_clinical_raw.csv"), index=False)
        print(f"  samples: {len(df)} | platforms: {df['platform'].value_counts().to_dict()}")
        summarize(df)
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")